<a href="https://colab.research.google.com/github/ljzier/ST-554-repo/blob/main/Zier_ST_554_HW10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Linda Zier**

**ST 554**

**HW #10**

## The Goal

The goal of this assignment is to get practice using spark structured streaming to deal with data.

## Part 1 - Creating Streaming Data Using rate
Setup a data stream using the "rate" format.
Prior to starting the stream, set up a sequence of actions using appropriate functions from pyspark.sql.functions
that uses the rate data to

• find the square root of the rate ‘value’

• find mod 4 of the rate ‘value’

To output this, create a writeStream that writes to ‘memory’ (format("memory")). Give the query a name
(queryName("...")) and start it!
Let it run for about 30 seconds and then stop the query. Then output the entire table stored in the query
name (spark.sql("select * from you_table_name").show()).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import time

spark = SparkSession.builder.getOrCreate()

# set up data stream with rate format
rateDF = spark.readStream.format("rate").load()

# find the sqrt of "value"
rateDF = rateDF.withColumn("sqrt_value", sqrt(col("value")))

# find mod 4 of "value"
rateDF = rateDF.withColumn("mod_value", mod(col("value"), 4))

# output to memory
writeDF = rateDF.writeStream.outputMode("append").format("memory").queryName("rate_query").start()

# run for 30 seconds
time.sleep(30)

#stop query
writeDF.stop()

# output table
spark.sql("select * from rate_query").show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 13:14:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Part 2 - Using data from a CSV with a Pipeline

There are six bikeDetails sub datasets available on the assignment link. The one named bikeDetails_for_fit.csv
should be read in as a spark (SQL) data frame. With this spark SQL data frame do the following

• use an SQLTransformer with the following statement (this does some log transforms, renames a variable, and creates a dummy variable from categorical variable):

```
SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
CASE WHEN owner = ’1st owner’ THEN 1 ELSE 0 END AS one_owner
FROM __THIS__
```
• use a VectorAssembler to create a features column. The features column should include the year,
log_km_driven, and one_owner variables.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

spark = SparkSession.builder.getOrCreate()

# read in bikeDetails_for_fit.csv as spqrk SQL dataframe
bikeDF=spark.read.csv("bikeDetails_for_fit.csv", header = "true', inferSchema="true")

# SQLTransformer
sqlTrans= SQLTransformer(statement =
                         '''
                         SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
                         CASE WHEN owner = ’1st owner’ THEN 1 ELSE 0 END AS one_owner
                         FROM __THIS__
                         ''')

#VectorAssembler to bundle features together
assembler = VectorAssembler(
    inputCols=['year', 'log_km_driven', 'one_owner'],
    outputCol='features')

print("transformations complete")


• create a Pipeline with the two steps above (SQLTransformer then VectorAssembler)

• fit this pipeline to the SQL data frame and save this as an object.

In [ ]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sqlTrans, assembler])
fitted_pipeline = pipeline.fit(bikeDF)

print("pipeline complete")


Now we want to set up a read stream to look for csv files placed into a folder (the five bikeDetails_add*.csv
files). When a csv comes in, we want to transform it using the fitted pipeline’s .transform() method! A
few notes:

• We need a schema to set up the readStream. We can use the SQL data frame’s schema from above!
(.schema attribute)

• Each file we’ll be adding to the folder has a header

In [ ]:
# setup a read stream
bikeStream = spark.readStream.schema(bikeDF.schema).option("header", True).csv("bike_files")

# transform new data
transStream = fitted_pipeline.transform(bikeStream)

